# 🧬 Remedia — Modal GPU Notebook

Bu notebook **Modal Notebooks** ve Remedia'nın özel Modal Jupyter başlatıcısı için hazırlanmıştır.

## Modal Notebooks ile

1. Sol taraftaki **Compute** bölümünden GPU olarak **L4** seç.
2. Kalıcı sonuç istiyorsan Files panelinden bir Volume oluşturup `/mnt/remedia-data` yoluna bağla.
3. **Run all** ile bütün hücreleri çalıştır.
4. Modal'ın idle shutdown süresini 10 dakika bırak; boşta kalınca ücret kesilir.

Volume bağlamazsan notebook yine çalışır, ancak kernel kapanınca sonuçlar silinir.


In [ ]:
# 1 — Ayarlar
UNIPROT_ID = "P00918"
BOX_DIM = 20

METHOD = "fusion"  # fusion | genetic | brics | random | pretrained
GENERATE_N = 20
GA_GENERATIONS = 3

DOCKING_MODE = "iki_asamali"  # iki_asamali | sadece_fast | sadece_accurate
ACCURACY_PROFILE = "balanced"  # balanced | final
TOP_FRACTION = 0.10

INSTALL_REINVENT4 = False
RUN_BENCHMARK = False
BENCHMARK_N = 3

USE_CACHED_POCKET = True
FORCE_REFRESH_POCKET = False

REPO_REF = "main"
UPDATE_REPO = False
PERSISTENT_ROOT = "/mnt/remedia-data"

print("✅ Ayarlar hazır")


In [ ]:
# 2 — Modal ortamını hazırla
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
import zipfile
from pathlib import Path

required_packages = {
    "requests": "requests>=2.31.0",
    "rdkit": "rdkit>=2023.9.1",
    "Bio": "biopython>=1.81",
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
    "scipy": "scipy>=1.11.0",
    "meeko": "meeko>=0.5.0",
    "gemmi": "gemmi>=0.6.5",
    "openbabel": "openbabel-wheel>=3.1.1",
    "tqdm": "tqdm>=4.66.0",
    "yaml": "pyyaml>=6.0",
    "PIL": "pillow>=10.0.0",
}
missing = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    print("📦 Eksik Python paketleri kuruluyor...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )

env_workspace = os.environ.get("REMEDIA_WORKSPACE")
preferred_root = Path(env_workspace or PERSISTENT_ROOT)
if preferred_root.exists():
    WORKSPACE = preferred_root.resolve()
    PERSISTENT = True
else:
    WORKSPACE = Path("/root/remedia_workspace").resolve()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    PERSISTENT = False
    print(
        "⚠️ Modal Volume bağlı değil. Bu oturum çalışır ama kernel kapanınca "
        "dosyalar silinir. Kalıcılık için /mnt/remedia-data yoluna Volume bağla."
    )

REPO_DIR = Path(os.environ.get("REMEDIA_HOME", WORKSPACE / "Remedia")).resolve()
TOOLS_DIR = WORKSPACE / ".remedia-tools"
CACHE_DIR = WORKSPACE / "remedia_cache"
RESULTS_ROOT = WORKSPACE / "Remedia_results"
for directory in (TOOLS_DIR / "bin", CACHE_DIR, RESULTS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def install_repo() -> None:
    if (REPO_DIR / "src").is_dir() and not UPDATE_REPO:
        return

    image_repo = Path("/opt/remedia")
    if (image_repo / "src").is_dir() and not UPDATE_REPO:
        print("⚡ Remedia özel Modal imajından kopyalanıyor...")
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        shutil.copytree(image_repo, REPO_DIR)
        return

    archive_url = (
        "https://github.com/mehmetg06/Remedia/archive/refs/heads/"
        f"{REPO_REF}.zip"
    )
    print(f"📥 Remedia {REPO_REF} indiriliyor...")
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        archive_path = tmp_path / "remedia.zip"
        urllib.request.urlretrieve(archive_url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(tmp_path)
        extracted = next(tmp_path.glob("Remedia-*"))
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        shutil.copytree(extracted, REPO_DIR)

def install_micromamba() -> Path:
    executable = TOOLS_DIR / "bin" / "micromamba"
    if executable.exists():
        return executable

    print("📦 micromamba kuruluyor...")
    url = "https://micro.mamba.pm/api/micromamba/linux-64/latest"
    with tempfile.TemporaryDirectory() as tmp:
        archive_path = Path(tmp) / "micromamba.tar.bz2"
        urllib.request.urlretrieve(url, archive_path)
        with tarfile.open(archive_path, "r:bz2") as archive:
            member = archive.getmember("bin/micromamba")
            member.name = "micromamba"
            archive.extract(member, TOOLS_DIR / "bin")
    executable.chmod(0o755)
    return executable

def install_fpocket(micromamba: Path) -> Path:
    executable = TOOLS_DIR / "fpocket" / "bin" / "fpocket"
    if executable.exists():
        return executable

    print("📦 fpocket kuruluyor...")
    subprocess.run(
        [
            str(micromamba),
            "create",
            "-y",
            "-p",
            str(TOOLS_DIR / "fpocket"),
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "fpocket",
        ],
        check=True,
    )
    return executable

def install_gnina() -> Path:
    image_binary = Path("/usr/local/bin/gnina")
    if image_binary.exists():
        return image_binary

    executable = TOOLS_DIR / "bin" / "gnina"
    if not executable.exists():
        print("📦 GNINA kuruluyor...")
        url = "https://github.com/gnina/gnina/releases/download/v1.3/gnina"
        urllib.request.urlretrieve(url, executable)
        executable.chmod(0o755)
    return executable

install_repo()
system_fpocket = Path("/opt/remedia-fpocket/bin/fpocket")
if system_fpocket.exists():
    FPOCKET_PATH = system_fpocket
else:
    micromamba_path = install_micromamba()
    FPOCKET_PATH = install_fpocket(micromamba_path)
GNINA_PATH = install_gnina()

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ["REMEDIA_HOME"] = str(REPO_DIR)
os.environ["REMEDIA_WORKSPACE"] = str(WORKSPACE)
os.environ["GNINA_PATH"] = str(GNINA_PATH)
os.environ["PATH"] = (
    str(FPOCKET_PATH.parent)
    + os.pathsep
    + str((TOOLS_DIR / "bin"))
    + os.pathsep
    + os.environ.get("PATH", "")
)

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError(
        "NVIDIA GPU görünmüyor. Modal Notebook Compute ayarından L4 seç."
    )

print("✅ Modal ortamı hazır")
print(f"GPU: {gpu.stdout.splitlines()[2] if len(gpu.stdout.splitlines()) > 2 else 'NVIDIA'}")
print(f"Repo: {REPO_DIR}")
print(f"Sonuçlar: {RESULTS_ROOT}")
print(f"Kalıcı depolama: {'evet' if PERSISTENT else 'hayır'}")


In [ ]:
# 3 — Hedef protein, pocket ve tohum moleküller
import datetime
import json
from pathlib import Path

RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULTS_DIR = RESULTS_ROOT / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_KEY = UNIPROT_ID.strip().upper()
BOX_SIZE = (float(BOX_DIM),) * 3
POCKET_CACHE_PATH = CACHE_DIR / "pocket_cache.json"

try:
    pocket_cache = (
        json.loads(POCKET_CACHE_PATH.read_text())
        if POCKET_CACHE_PATH.exists()
        else {}
    )
except Exception:
    pocket_cache = {}

cached = (
    pocket_cache.get(CACHE_KEY)
    if USE_CACHED_POCKET and not FORCE_REFRESH_POCKET
    else None
)

import fetch_structure
import pocket_detection

pdb_path = fetch_structure.fetch_alphafold(CACHE_KEY)
RECEPTOR = str(pdb_path)

if cached:
    CENTER = tuple(float(x) for x in cached["center"])
    print(f"⚡ Pocket cache kullanıldı: {CENTER}")
else:
    pocket = pocket_detection.best_druggable_pocket(pdb_path)
    CENTER = tuple(round(float(x), 3) for x in pocket["center"])
    pocket_cache[CACHE_KEY] = {
        "center": list(CENTER),
        "source": "fpocket",
        "druggability": pocket.get("druggability"),
    }
    POCKET_CACHE_PATH.write_text(json.dumps(pocket_cache, indent=2))
    print(f"✅ fpocket merkezi: {CENTER}")

from known_ligands import fetch_known_ligands
from rdkit import Chem

ligands, message = fetch_known_ligands(CACHE_KEY, max_results=5)
print(message)
seeds = [item["smiles"] for item in ligands]
if not seeds:
    seeds = [
        "NS(=O)(=O)c1ccccc1",
        "CC(=O)Nc1nnc(s1)S(N)(=O)=O",
    ]
seeds = [smiles for smiles in seeds if Chem.MolFromSmiles(smiles)]
if not seeds:
    raise RuntimeError("Geçerli tohum molekül bulunamadı.")

print(f"✅ Hedef={CACHE_KEY} · center={CENTER} · seed={len(seeds)}")
print(f"📁 Sonuç klasörü: {RESULTS_DIR}")


In [ ]:
# 4 — Molekül üretimi ve batch GNINA docking
import pandas as pd

from molecule_generator import (
    brics_recombination,
    fusion_generation,
    genetic_algorithm,
    random_mutation,
    write_smi,
)

method = METHOD.lower().strip()
if method == "pretrained":
    if not INSTALL_REINVENT4:
        raise RuntimeError(
            "METHOD='pretrained' için INSTALL_REINVENT4=True yap."
        )
    from generative_model import install_reinvent, generate_with_reinvent

    install_reinvent(log_fn=print, drive_cache_dir=CACHE_DIR)
    generated = generate_with_reinvent(
        num_molecules=GENERATE_N,
        output_path=RESULTS_DIR / "generated.smi",
        drive_cache_dir=CACHE_DIR,
    )
elif method == "fusion":
    final, _ = fusion_generation(
        seeds,
        docking_opts=None,
        log_fn=print,
        population_size=max(10, GENERATE_N),
        generations=GA_GENERATIONS,
    )
    generated = [smiles for smiles, _score in final]
elif method == "genetic":
    final, _ = genetic_algorithm(
        seeds,
        generations=GA_GENERATIONS,
        population_size=max(10, GENERATE_N),
        docking_opts=None,
        log_fn=print,
    )
    generated = [smiles for smiles, _score in final]
elif method == "brics":
    generated = brics_recombination(seeds, n=GENERATE_N)
elif method == "random":
    generated = random_mutation(seeds, n=GENERATE_N)
else:
    raise ValueError(f"Bilinmeyen METHOD: {METHOD}")

if method != "pretrained" and len(generated) < GENERATE_N:
    pool = set(generated)
    pool.update(random_mutation(seeds, n=GENERATE_N))
    pool.update(brics_recombination(seeds, n=GENERATE_N))
    generated = list(pool)

generated = [smiles for smiles in generated if smiles][:GENERATE_N]
if not generated:
    raise RuntimeError("Molekül üretilemedi.")

molecules = [
    (f"mol_{index:03d}", smiles)
    for index, smiles in enumerate(generated, 1)
]
smi_out = RESULTS_DIR / "generated.smi"
if method != "pretrained":
    write_smi(generated, smi_out, prefix="mol")

import gnina_engine

common = dict(
    receptor=RECEPTOR,
    center=CENTER,
    size=BOX_SIZE,
    gnina_path=str(GNINA_PATH),
    out_dir=RESULTS_DIR / "gnina_out",
    log_fn=print,
    profile=ACCURACY_PROFILE,
)

if DOCKING_MODE == "iki_asamali":
    rows, stage_info = gnina_engine.run_two_stage_screening(
        molecules,
        top_fraction=TOP_FRACTION,
        **common,
    )
elif DOCKING_MODE == "sadece_fast":
    rows, stage_info = gnina_engine.run_single_mode_screening(
        molecules,
        mode="fast",
        **common,
    )
elif DOCKING_MODE == "sadece_accurate":
    rows, stage_info = gnina_engine.run_single_mode_screening(
        molecules,
        mode="accurate",
        **common,
    )
else:
    raise ValueError(f"Bilinmeyen DOCKING_MODE: {DOCKING_MODE}")

docking_df = pd.DataFrame(rows)
display(docking_df)
print("✅ GNINA süreç sayısı:", stage_info.get("gnina_processes"))

if RUN_BENCHMARK:
    bench_rows, summary = gnina_engine.benchmark_fast_vs_accurate(
        molecules[:BENCHMARK_N],
        receptor=RECEPTOR,
        center=CENTER,
        size=BOX_SIZE,
        gnina_path=str(GNINA_PATH),
        out_dir=RESULTS_DIR / "benchmark",
        profile=ACCURACY_PROFILE,
    )
    pd.DataFrame(bench_rows).to_csv(
        RESULTS_DIR / "benchmark.csv",
        index=False,
    )
    print(summary)
else:
    print("⏭️ Benchmark kapalı")


In [ ]:
# 5 — Drug-likeness filtresi, sıralama ve kayıt
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import FileLink, display

import admet_filter

admet_df = pd.DataFrame(
    [
        admet_filter.lipinski_veber_filter(smiles, name)
        for name, smiles in molecules
    ]
)

docking_csv = RESULTS_DIR / "docking_scores.csv"
admet_csv = RESULTS_DIR / "admet_results.csv"
ranked_csv = RESULTS_DIR / "final_ranking.csv"

docking_df.to_csv(docking_csv, index=False)
admet_df.to_csv(admet_csv, index=False)

subprocess.run(
    [
        sys.executable,
        str(SRC_DIR / "rank_report.py"),
        "--docking",
        str(docking_csv),
        "--admet",
        str(admet_csv),
        "--output",
        str(ranked_csv),
    ],
    check=True,
)

ranked_df = pd.read_csv(ranked_csv)
display(ranked_df.head(10))

archive = shutil.make_archive(
    str(RESULTS_DIR),
    "zip",
    root_dir=RESULTS_DIR,
)
print(f"✅ Tamamlandı: {RESULTS_DIR}")
display(FileLink(archive))
